Packages

In [ ]:
# Public packages
import math
import os
import re
from tqdm import tqdm
import tabulate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown
from pathlib import Path

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Preemptively set new Pandas option, increase display max rows, allow matplotlib visuals, and set matplotlib to close
pd.options.mode.copy_on_write = True
pd.set_option('display.max_rows', 100)
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
%load_ext autoreload
%autoreload 2

# Load formatted data if it's there 
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files (ignore details)

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
# Data already exists
else:
    static_data = static_data_merged.copy()

%store static_data_merged

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

%store sales_data_merged

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/3_data_parquet_relabeled/before_after_details_true.csv', index_col='location_id')
%store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/3_data_parquet_relabeled/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
%store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/3_data_parquet_relabeled/restaurants_by_4m_coverage.csv')['location_id'].tolist()
%store restaurants_by_4m_coverage

loc_id = 'W8T41JZK0ZMEP'
df = sales_and_menu_data[loc_id]

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
df.head()

In [ ]:
df['item_name'].value_counts()

In [ ]:
# Consolidate dishes
name_changes = {
    "Impossible Patty Melt" : ["Impossible Melt"],
    "Gold Standard - Impossible" : ["Gold Standard Impossible", "Imp Breakfast"], 
    "Gold Standard - Bacon" : ["Gold Standard - Bacon!", "Gold Standard Breakfast Sandwich"],
    "Gold Standard - Bacon & Kale" : ["Gold Standard - Bacon/Kale", "Gold Standard - Bacon + Kale", "Both - Gold Standard", "Gold Standard - Bacon Ü•Ì"],  
    "Telway Burger" : ["Telway"]
    }

# Create a dictionary by swapping the keys and values
name_changes_dict = {variant: canonical for canonical, variants in name_changes.items() for variant in variants}

# Rename items based on modications
modification_name_changes = [
    ('Gold Standard - Impossible', 'Add Bacon|Extra Bacon|Regular Bacon|Beef', 'Gold Standard - Bacon/Beef & Impossible'),
    ('Beyond Burger', 'Add Bacon|Beef', 'Bacon/Beef Beyond Burger'),
    ('Beyond Burger', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Beyond Burger'),
    ('Impossible Patty Melt', 'Add Bacon|Beef|Meat', 'Bacon/Beef Impossible Patty Melt'),
    ('Impossible Patty Melt', 'No Cheese|No Chz|No C|No Cuz|Vegan', 'Vegan Impossible Patty Melt'),
    ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Sub Impossible', 'Gold Standard - Impossible'),
    ('Gold Standard - Bacon|Gold Standard - Bacon & Kale', 'Add Impossible', 'Gold Standard - Bacon & Impossible'),
    ('Fresh Beyond Burger', 'Bacon', 'Beyond Burger With Bacon') ,
    ('Fresh Beyond Burger', 'Cheddar Cheese', 'Beyond Burger With Dairy'),
    ('Beyond Burger Combo', 'Bacon', 'Beyond Burger Combo With Bacon'),
    ('Beyond Burger Combo', 'Cheddar Cheese', 'Beyond Burger Combo With Dairy')]

# Create a dataframe
modification_name_changes_df = pd.DataFrame(data = modification_name_changes, columns = ['name', 'modification', 'new_name'])

merch = []

alcoholic_drinks = []

non_alcoholic_drinks = []

vegan = ['Fresh Beyond Burger','Beyond Burger Combo']

vegetarian = ['Beyond Burger With Dairy', 
              'Beyond Burger Combo With Dairy']

meat = ['Gold Standard - Bacon/Beef & Impossible']

rare =[]

unknown = []

# The four commands:

### 1. All items:
print(df['item_name'].value_counts().to_string())

### 2. Non merch, non drink, non alcoholic, items labeled as plant-based:
print(df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "Yes"')['item_name'].value_counts().to_string())

### 3. Modifications for a specific item:
print(df.query('item_name == "Kale Caesar Salad"')['item_modifications'].value_counts().to_string())

### 4. Modifications for a specific item filtered to contain a string:
df.query('item_name == "Kale Caesar Salad"')['item_modifications'].value_counts().filter(regex='Chicken')


In [ ]:
df.query('item_name == "Pb & J"')['item_modifications'].value_counts()

In [ ]:
print(df['item_name'].value_counts().to_string())

In [ ]:
# 1. Relabeling dishes by name
# 'yes' -> vegetarian, meat
# 'unsure' -> vegan, vegetarian, meat
# 'no' -> vegan, vegetarian

# 2. Relabeling dishes by modification

# 3. Collapse dishes into menu items

In [ ]:
df.query('item_name == "Breakfast Supreme"')['item_modifications'].value_counts()

In [ ]:
print(df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "Yes"')['item_name'].value_counts().to_string())

In [ ]:
# List the item modifications for a given item
# Paste the item name right below THIS space 
# Buffalo "Tofu Chicken" Wrap - invalid format
print(df.query("item_name == '''Avocado Toast'''")['item_modifications'].value_counts().to_string())

In [ ]:
# Tinker with regex to check whether a string is in the item modifications of an item
print(df.query('item_name == "Avocado Toast"')['item_modifications'].value_counts().filter(regex='Vegan').to_string())